# Module 9: RAG & LLM Applications
## 04: Systematic Prompt Engineering & Advanced Reasoning

### Theoretical Overview
Prompt engineering is systematic communication with autoregressive language models. Key techniques include:
- **Role & Context Framing**: Establishing expert identity, domain constraints, and audience profile.
- **Delimiter Isolation**: Encapsulating user input with XML tags (`<context>`, `<query>`) to prevent prompt injection.
- **Few-Shot In-Context Demonstrations**: Providing exemplar input-output pairs to anchor the formatting distribution.
- **Chain-of-Thought (CoT)**: Forcing the model to generate intermediate reasoning tokens before emitting conclusions.

In [ ]:
class PromptBuilder:
    """Constructs structured, injection-resistant prompt templates."""
    def __init__(self, system_role: str):
        self.system_role = system_role
        self.few_shot_examples = []
        
    def add_example(self, user_in: str, assistant_out: str):
        self.few_shot_examples.append({"user": user_in, "assistant": assistant_out})
        return self
        
    def build(self, user_query: str, context: Optional[str] = None) -> str:
        prompt_parts = [f"SYSTEM: {self.system_role}
"]
        if self.few_shot_examples:
            prompt_parts.append("### EXAMPLES ###")
            for ex in self.few_shot_examples:
                prompt_parts.append(f"Q: {ex['user']}
A: {ex['assistant']}")
            prompt_parts.append("### END EXAMPLES ###
")
            
        if context:
            prompt_parts.append(f"<context>
{context}
</context>
")
            
        prompt_parts.append(f"<user_query>
{user_query}
</user_query>")
        prompt_parts.append("
REASONING STEP-BY-STEP:")
        return "
".join(prompt_parts)

builder = (
    PromptBuilder("You are an expert medical coder translating clinical notes into ICD-10 codes.")
    .add_example("Patient presents with acute pharyngitis.", "ICD-10: J02.9 (Acute pharyngitis, unspecified)")
    .add_example("Diagnosed type 2 diabetes mellitus without complications.", "ICD-10: E11.9")
)

final_prompt = builder.build("Patient displays symptoms of essential primary hypertension.", context="Chart notes: BP 145/95")
print(final_prompt)

### Chain-of-Thought vs Direct Output

```
Direct Prompting:
"Calculate total profit for 420 units at $15 sale price, with 30% overhead and $8 unit cost."
-> Error-prone single-pass estimation.

CoT Prompting:
"Break down the computation step-by-step:
1. Revenue = units * price
2. Direct costs = units * cost
3. Overhead = 30% of revenue
4. Profit = Revenue - Direct costs - Overhead"
-> Produces verified intermediate tokens, dramatically reducing numerical errors.
```

In [ ]:
def simulate_cot_reasoning(units: int, price: float, unit_cost: float, overhead_rate: float):
    revenue = units * price
    direct_cost = units * unit_cost
    overhead = revenue * overhead_rate
    profit = revenue - direct_cost - overhead
    
    cot_trace = [
        f"1. Revenue: {units} * ${price:.2f} = ${revenue:,.2f}",
        f"2. Direct Costs: {units} * ${unit_cost:.2f} = ${direct_cost:,.2f}",
        f"3. Overhead: ${revenue:,.2f} * {overhead_rate:.0%} = ${overhead:,.2f}",
        f"4. Net Profit: ${revenue:,.2f} - ${direct_cost:,.2f} - ${overhead:,.2f} = ${profit:,.2f}"
    ]
    return "
".join(cot_trace)

print(simulate_cot_reasoning(420, 15.0, 8.0, 0.30))